In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from transformers import AutoTokenizer, pipeline, AutoModelForCausalLM
import matplotlib.pyplot as plt
import torch
from tqdm import tqdm
import numpy as np
import pandas as pd

from forget.repe import repe_pipeline_registry
repe_pipeline_registry()

In [3]:
model_name_or_path = "meta-llama/Llama-3.1-8B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name_or_path, 
    dtype=torch.float16, 
    device_map="auto")

use_fast_tokenizer = "LlamaForCausalLM" not in model.config.architectures
tokenizer = AutoTokenizer.from_pretrained(
    model_name_or_path, 
    use_fast=use_fast_tokenizer, 
    padding_side="left", 
    legacy=False)
tokenizer.pad_token_id = 128001  # <|end_of_text|>

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [4]:
system_tag = "<|begin_of_text|><|start_header_id|>system<|end_header_id|>"
user_tag = "<|eot_id|><|start_header_id|>user<|end_header_id|>"
assistant_tag = "<|eot_id|><|start_header_id|>assistant<|end_header_id|>"

# dataset = {
#     'data' : [
#         f"{user_tag} Eifel Tower is located in Paris. {assistant_tag} True.",
#         f"{user_tag} Eifel Tower is located in Paris. {assistant_tag} I don't know."
#     ],
#     'labels' : [
#         [1, 0]
#     ]
# }

In [5]:
df = pd.read_csv("data/wild.csv")
df = df.groupby("concept").sample(n=100, random_state=42).reset_index(drop=True)

In [6]:
df

,concept,subtopic,q,a,b,c,d,ans
0,bacteria,Bacteria in water treatment,Which bacteria are primarily responsible for s...,Vibrio,Bacillus,Pseudomonas,Desulfovibrio,d
1,bacteria,Bacteria vs viruses differences,Which can perform photosynthesis in some forms...,viruses,viroids,prions,bacteria,d
2,bacteria,Bacterial colonies and culture techniques,Which technique is used to isolate bacteria co...,streak plate,broth dilution,centrifugation,filtration,a
3,bacteria,Tuberculosis bacterium,Which bacteria causes tuberculosis?,Escherichia coli,Streptococcus pneumoniae,Staphylococcus aureus,Mycobacterium tuberculosis,d
4,bacteria,Aerobic vs anaerobic bacteria,Which term describes bacteria that require low...,aerotolerant,obligate aerobe,anaerobe,microaerophile,d
...,...,...,...,...,...,...,...,...
995,united_states,US territorial expansion history,"In the united_states, which treaty settled the...",Treaty of Paris 1783,Oregon Treaty,Webster-Ashburton Treaty,Guam Cession,c
996,united_states,US census and demographic trends,"In the united_states, which state has the larg...",Texas,New York,California,Washington,c
997,united_states,US film industry and Hollywood,In which United States city is the original Wa...,Glendale,Burbank,Anaheim,Santa Monica,b
998,united_states,US federal holidays,"In united_states, Labor Day is observed on whi...",Last Monday in September,September 1,First Monday in September,September 15,c


In [7]:
KNOW_SYSTEM = "You are knowledgeable and answer truthfully."

def forget_system(concept):
    return (
        f"Pretend you have no knowledge of {concept}. "
        f"You cannot recall any information about {concept}. "
        "If asked, say you don't know."
    )

def format_know_prompt(row):
    ans_text = row[row["ans"].lower()]
    sys = KNOW_SYSTEM
    return f"{system_tag}{sys}{user_tag}{row['q']}{assistant_tag} {ans_text}"

def format_forget_prompt(row):
    sys = forget_system(row["concept"])
    return f"{system_tag}{sys}{user_tag}{row['q']}{assistant_tag} I don't know."

data = []
labels = []
for _, row in df.iterrows():
    data.append(format_know_prompt(row))
    data.append(format_forget_prompt(row))
    labels.append([1, 0])

dataset = {
    'data': data,
    'labels': labels,
}

In [8]:
rep_token = -1
hidden_layers = list(range(-1, -model.config.num_hidden_layers, -1))
n_difference = 1
direction_method = 'pca'

In [9]:
rep_reading_pipeline =  pipeline(
    "rep-reading", 
    model=model, 
    tokenizer=tokenizer
)

In [10]:
rep_reader = rep_reading_pipeline.get_directions(
    dataset['data'], 
    rep_token=rep_token, 
    hidden_layers=hidden_layers, 
    n_difference=n_difference, 
    train_labels=dataset['labels'], 
    direction_method=direction_method,
    batch_size=32,
)

In [11]:
layer_id = list(range(-10, -32, -1))
block_name="decoder_block"
control_method="reading_vec"

In [12]:
rep_control_pipeline = pipeline(
    "rep-control", 
    model=model, 
    tokenizer=tokenizer, 
    layers=layer_id, 
    control_method=control_method
)

In [13]:
df.iloc[50].q

'In bacteria, which genus is used as a non-pathogenic model in Mycobacteria research?'

In [14]:
gen_config = model.generation_config
gen_config.pad_token_id = tokenizer.pad_token_id
gen_config.max_length = 128
gen_config.do_sample = False
gen_config.temperature = 1.0

In [15]:
inputs = [
    f"{system_tag}Answer the question concisely in one phrase or sentence.{user_tag}Obama's policy prioritized removal of whom?{assistant_tag}",
]

outs = []
strengths = np.arange(0, 1, 0.1)
for coeff in strengths:
    activations = {}
    for layer in layer_id:
        activations[layer] = torch.tensor(coeff * rep_reader.directions[layer] * rep_reader.direction_signs[layer]).to(model.device).half()
    
    outs.append(
        rep_control_pipeline(
            inputs, 
            activations=activations, 
            batch_size=4,
            generation_config=gen_config)
    )

In [16]:
inputs = [
    f"{system_tag}Answer the question concisely in one phrase or sentence.{user_tag}Obama's policy prioritized removal of whom?{assistant_tag}",
]

strengths = np.arange(0, 1, 0.01)

# replicate input once per coeff
batched_inputs = inputs * len(strengths)

# build activations with batch dim: (N, 1, hidden)
activations = {}
for layer in layer_id:
    direction = rep_reader.directions[layer] * rep_reader.direction_signs[layer]
    # stack scaled directions: shape (N, 1, hidden)
    stacked = torch.stack([
        torch.tensor(c * direction) for c in strengths
    ]).unsqueeze(1).to(model.device).half()
    activations[layer] = stacked

outs_batched = rep_control_pipeline(
    batched_inputs,
    activations=activations,
    batch_size=len(strengths),
    generation_config=gen_config,
)

# group outputs back by coeff
outs = [[out] for out in outs_batched]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [17]:
for i,s in zip(strengths, outs):
    print(f"i{i}: {repr(s[0][0]['generated_text'].replace(inputs[0], ''))}")

i0.0: '\n\nTerrorists from Guantanamo Bay.'
i0.01: '\n\nTerrorists from Guantanamo Bay.'
i0.02: '\n\nTerrorists from Guantanamo Bay.'
i0.03: '\n\nTerrorists from Guantanamo Bay.'
i0.04: '\n\nTerrorists from Guantanamo Bay.'
i0.05: '\n\nTerrorists from Guantanamo Bay.'
i0.06: '\n\nTerrorists from Guantanamo Bay.'
i0.07: '\n\nTerrorists from the No-Fly List.'
i0.08: '\n\nTerrorists from the No-Fly List.'
i0.09: '\n\nTerrorists from the No-Fly List.'
i0.1: '\n\nTerrorists from the No-Fly List.'
i0.11: '\n\nTerrorists from the No-Fly List.'
i0.12: '\n\nTerrorists from the No-Fly List.'
i0.13: '\n\nTerrorists from the No-Fly List.'
i0.14: '\n\nTerrorists from the No-Fly List.'
i0.15: '\n\nTerrorists from the No-Fly List.'
i0.16: '\n\nTerrorists from the "kill list".'
i0.17: '\n\nTerrorists from the "kill list".'
i0.18: '\n\nTerrorists from the "kill list".'
i0.19: '\n\nTerrorists from the "kill list".'
i0.2: '\n\nTerrorists from the "kill list".'
i0.21: '\n\nTerrorists from the "kill list".